# Phase 2 QRC Leaky-Input Trajectory Probe

The ESN-vs-QRC failure analysis showed that the ESN resets per 40-day window, yet still strongly outperforms QRC. This points away from cross-window carryover as the immediate cause and toward **within-window trajectory processing**.

This notebook tests whether QRC improves when each PCA window is leaky-integrated before anchor encoding. The input dimension stays PCA-6; the reservoir architecture and readout stay fixed.

Base QRC setting:

- lookback_days = 40
- anchor_count = 10
- anchor_policy = recent
- full TFIM topology
- 3 Trotter steps per anchor
- 3 virtual nodes per anchor
- evolution_time = 0.5
- ZXZZ observables
- disorder_strength = 0.20
- winsorized top-120 readout
- ridge alpha = 3000

Carryover/streaming QRC is left as an extension, not the next step, because reset-window ESN already beats reset-window QRC.

In [1]:
from pathlib import Path
import os

if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

from qpitome_qrc.data.features import FEATURE_COLUMNS
from qpitome_qrc.data.loaders import load_phase2_volatility_data
from qpitome_qrc.data.pca import fit_transform_pca_splits_train_only
from qpitome_qrc.data.splits import chronological_tabular_split
from qpitome_qrc.evaluation.metrics import evaluate_volatility_forecast
from qpitome_qrc.qrc.tfim_reservoir import (
    TFIMQRCConfig,
    _safe_feature_target_correlations,
    diagnose_reservoir_feature_splits,
    fit_tfim_qrc_regressor,
    make_qrc_sequence_splits,
)

## 1. Data and sequence splits

In [2]:
target = "future_rv_20d"

df = load_phase2_volatility_data()
splits = chronological_tabular_split(df)

pca6 = fit_transform_pca_splits_train_only(
    splits,
    feature_columns=FEATURE_COLUMNS,
    target_columns=[target],
    n_components=6,
    prefix="pca6",
)

raw_seq = make_qrc_sequence_splits(
    pca6.splits,
    feature_columns=pca6.feature_columns,
    target_column=target,
    lookback_days=40,
)

print({k: (v[0].shape, v[1].shape) for k, v in raw_seq.items()})

{'train': ((5420, 40, 6), (5420,)), 'val': ((1219, 40, 6), (1219,)), 'test': ((1019, 40, 6), (1019,))}


## 2. Leaky-window preprocessing

For each independent 40-day window, compute a classical leaky-integrated trajectory:

```text
h_t = (1 - leak) h_{t-1} + leak x_t
```

The QRC then receives anchors from `h_t` instead of anchors from raw PCA state `x_t`. This gives each anchor a trajectory-aware summary while preserving PCA-6 input dimension.

In [3]:
def leaky_integrate_window(window, leak):
    h = np.zeros(window.shape[1], dtype=float)
    out = np.zeros_like(window, dtype=float)
    for t, u_t in enumerate(window):
        h = (1.0 - leak) * h + leak * u_t
        out[t] = h
    return out


def leaky_integrate_windows(X, leak):
    return np.stack([leaky_integrate_window(window, leak) for window in X], axis=0)


def transform_sequence_splits(sequence_splits, leak):
    if leak is None:
        return sequence_splits
    return {
        split: (leaky_integrate_windows(X, leak), y, dates)
        for split, (X, y, dates) in sequence_splits.items()
    }

## 3. Readout and diagnostics helpers

In [4]:
def robust_topk_readout(result, sequence_splits, *, top_k=120, alpha=3000.0):
    H_train = result.train_features
    H_val = result.val_features
    H_test = result.test_features

    _, y_train, _ = sequence_splits["train"]
    _, y_val, _ = sequence_splits["val"]
    _, y_test, _ = sequence_splits["test"]

    lower = np.percentile(H_train, 1.0, axis=0)
    upper = np.percentile(H_train, 99.0, axis=0)
    H_train = np.clip(H_train, lower, upper)
    H_val = np.clip(H_val, lower, upper)
    H_test = np.clip(H_test, lower, upper)

    corr = _safe_feature_target_correlations(H_train, y_train)
    k = min(top_k, H_train.shape[1])
    idx = np.argsort(np.abs(corr))[-k:]
    H_train = H_train[:, idx]
    H_val = H_val[:, idx]
    H_test = H_test[:, idx]

    scaler = StandardScaler()
    H_train_s = scaler.fit_transform(H_train)
    H_val_s = scaler.transform(H_val)
    H_test_s = scaler.transform(H_test)

    model = Ridge(alpha=alpha)
    model.fit(H_train_s, np.log(np.maximum(y_train, 1e-8)))

    return {
        "pred_train": np.exp(model.predict(H_train_s)),
        "pred_val": np.exp(model.predict(H_val_s)),
        "pred_test": np.exp(model.predict(H_test_s)),
        "H_train": H_train,
        "H_val": H_val,
        "H_test": H_test,
        "selected_idx": idx,
        "top_k_used": k,
    }


def split_metrics(y_train, y_val, y_test, pred_train, pred_val, pred_test):
    out = {}
    for name, y, pred in [
        ("train", y_train, pred_train),
        ("val", y_val, pred_val),
        ("test", y_test, pred_test),
    ]:
        m = evaluate_volatility_forecast(y, pred)
        out[f"{name}_rmse"] = m.rmse
        out[f"{name}_qlike"] = m.qlike
        out[f"{name}_mz_r2"] = m.mz_r2
        out[f"{name}_mz_beta"] = m.mz_beta
        out[f"{name}_pred_mean"] = float(np.mean(pred))
        out[f"{name}_pred_std"] = float(np.std(pred))
        out[f"{name}_corr"] = float(np.corrcoef(y, pred)[0, 1])
    return out


def high_vol_stats(y_train, y, pred, q=0.80):
    threshold = np.quantile(y_train, q)
    actual_high = y >= threshold
    pred_high = pred >= threshold
    tp = int(np.sum(actual_high & pred_high))
    fp = int(np.sum(~actual_high & pred_high))
    fn = int(np.sum(actual_high & ~pred_high))
    return {
        "threshold": float(threshold),
        "actual_high_rate": float(actual_high.mean()),
        "pred_high_rate": float(pred_high.mean()),
        "high_vol_recall": tp / max(tp + fn, 1),
        "high_vol_precision": tp / max(tp + fp, 1),
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }

## 4. Raw vs leaky-integrated QRC runs

In [5]:
input_configs = [
    {"run_name": "raw_anchor10_recent", "leak": None},
    {"run_name": "leaky03_anchor10_recent", "leak": 0.3},
    {"run_name": "leaky05_anchor10_recent", "leak": 0.5},
    {"run_name": "leaky07_anchor10_recent", "leak": 0.7},
]

rows = []
diag_rows = []
hv_rows = []

for cfg in input_configs:
    print("Running", cfg["run_name"])
    seq = transform_sequence_splits(raw_seq, cfg["leak"])

    _, y_train, _ = seq["train"]
    _, y_val, _ = seq["val"]
    _, y_test, _ = seq["test"]

    qrc_config = TFIMQRCConfig(
        qubits=6,
        pca_components=6,
        lookback_days=40,
        anchor_count=10,
        anchor_policy="recent",
        observable_mode="zxzz",
        collect_anchor_features=True,
        topology="full",
        trotter_steps_per_anchor=3,
        virtual_nodes_per_anchor=3,
        coupling_scale=0.7,
        transverse_field=0.5,
        evolution_time=0.5,
        angle_max=np.pi / 2,
        ridge_alpha=3000.0,
        target_transform="log",
        seed=42,
        use_disorder=True,
        disorder_strength=0.20,
    )

    base_result = fit_tfim_qrc_regressor(seq, config=qrc_config, target=target, verbose=True)
    readout = robust_topk_readout(base_result, seq, top_k=120, alpha=3000.0)

    row = {
        "run_name": cfg["run_name"],
        "leak": cfg["leak"],
        "anchor_count": 10,
        "anchor_policy": "recent",
        "n_raw_features": base_result.train_features.shape[1],
        "n_selected_features": readout["top_k_used"],
    }
    row.update(split_metrics(
        y_train, y_val, y_test,
        readout["pred_train"], readout["pred_val"], readout["pred_test"],
    ))
    row.update({f"test_{k}": v for k, v in high_vol_stats(y_train, y_test, readout["pred_test"]).items()})
    rows.append(row)

    diag = diagnose_reservoir_feature_splits(
        readout["H_train"], readout["H_val"], readout["H_test"],
        y_train, y_val, y_test,
    )
    diag.insert(0, "run_name", cfg["run_name"])
    diag.insert(1, "leak", cfg["leak"] if cfg["leak"] is not None else "raw")
    diag_rows.append(diag)

    for split_name, y, pred in [
        ("train", y_train, readout["pred_train"]),
        ("val", y_val, readout["pred_val"]),
        ("test", y_test, readout["pred_test"]),
    ]:
        hv = high_vol_stats(y_train, y, pred)
        hv.update({"run_name": cfg["run_name"], "leak": cfg["leak"] if cfg["leak"] is not None else "raw", "split": split_name})
        hv_rows.append(hv)

leaky_results = pd.DataFrame(rows)
leaky_diagnostics = pd.concat(diag_rows, ignore_index=True)
leaky_high_vol = pd.DataFrame(hv_rows)

leaky_results.sort_values(["test_rmse", "test_qlike"], ascending=[True, True])

Running raw_anchor10_recent
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/5420
QRC sample 3500/5420
QRC sample 3750/5420
QRC sample 4000/5420
QRC sample 4250/5420
QRC sample 4500/5420
QRC sample 4750/5420
QRC sample 5000/5420
QRC sample 5250/5420
QRC sample 0/1219
QRC sample 250/1219
QRC sample 500/1219
QRC sample 750/1219
QRC sample 1000/1219
QRC sample 0/1019
QRC sample 250/1019
QRC sample 500/1019
QRC sample 750/1019
QRC sample 1000/1019
Running leaky03_anchor10_recent
QRC sample 0/5420
QRC sample 250/5420
QRC sample 500/5420
QRC sample 750/5420
QRC sample 1000/5420
QRC sample 1250/5420
QRC sample 1500/5420
QRC sample 1750/5420
QRC sample 2000/5420
QRC sample 2250/5420
QRC sample 2500/5420
QRC sample 2750/5420
QRC sample 3000/5420
QRC sample 3250/542

,run_name,leak,anchor_count,anchor_policy,n_raw_features,n_selected_features,train_rmse,train_qlike,train_mz_r2,train_mz_beta,...,test_pred_std,test_corr,test_threshold,test_actual_high_rate,test_pred_high_rate,test_high_vol_recall,test_high_vol_precision,test_tp,test_fp,test_fn
1,leaky03_anchor10_recent,0.3,10,recent,459,120,0.078307,-2.561660,0.401142,1.183570,...,0.050512,0.407943,0.213021,0.23945,0.240432,0.549180,0.546939,134,111,110
2,leaky05_anchor10_recent,0.5,10,recent,459,120,0.080551,-2.539251,0.366462,1.194700,...,0.048302,0.401318,0.213021,0.23945,0.237488,0.504098,0.508264,123,119,121
3,leaky07_anchor10_recent,0.7,10,recent,459,120,0.082506,-2.514568,0.337709,1.223770,...,0.045265,0.381633,0.213021,0.23945,0.214917,0.434426,0.484018,106,113,138
0,raw_anchor10_recent,NaN,10,recent,459,120,0.084755,-2.479928,0.306357,1.286031,...,0.041984,0.344242,0.213021,0.23945,0.174681,0.352459,0.483146,86,92,158


## 5. High-volatility recall view

In [6]:
leaky_high_vol.sort_values(["split", "high_vol_recall"], ascending=[True, False])

,threshold,actual_high_rate,pred_high_rate,high_vol_recall,high_vol_precision,tp,fp,fn,run_name,leak,split
5,0.213021,0.239450,0.240432,0.549180,0.546939,134,111,110,leaky03_anchor10_recent,0.3,test
8,0.213021,0.239450,0.237488,0.504098,0.508264,123,119,121,leaky05_anchor10_recent,0.5,test
11,0.213021,0.239450,0.214917,0.434426,0.484018,106,113,138,leaky07_anchor10_recent,0.7,test
2,0.213021,0.239450,0.174681,0.352459,0.483146,86,92,158,raw_anchor10_recent,raw,test
3,0.213021,0.200000,0.157749,0.489852,0.621053,531,324,553,leaky03_anchor10_recent,0.3,train
6,0.213021,0.200000,0.150738,0.455720,0.604651,494,323,590,leaky05_anchor10_recent,0.5,train
9,0.213021,0.200000,0.133948,0.391144,0.584022,424,302,660,leaky07_anchor10_recent,0.7,train
0,0.213021,0.200000,0.102214,0.273063,0.534296,296,258,788,raw_anchor10_recent,raw,train
10,0.213021,0.105004,0.023790,0.070312,0.310345,9,20,119,leaky07_anchor10_recent,0.7,val
4,0.213021,0.105004,0.040197,0.054688,0.142857,7,42,121,leaky03_anchor10_recent,0.3,val


## 6. Selected-feature diagnostics

In [7]:
leaky_diagnostics.sort_values(["run_name", "split"])

,run_name,leak,split,n_samples,n_features,near_constant_features,feature_std_min,feature_std_median,feature_std_max,effective_rank,condition_number,mean_abs_feature_target_corr,max_abs_feature_target_corr,mean_abs_shift_vs_train,max_abs_shift_vs_train
5,leaky03_anchor10_recent,0.3,test,1019,120,0,0.123204,0.236416,0.406016,45.980535,1000.025933,0.287461,0.440467,0.231152,0.408986
3,leaky03_anchor10_recent,0.3,train,5420,120,0,0.091946,0.221916,0.419280,47.465545,890.478122,0.347662,0.498115,0.000000,0.000000
4,leaky03_anchor10_recent,0.3,val,1219,120,0,0.077632,0.215665,0.348437,46.982992,1037.650967,0.140980,0.370549,0.201814,0.508016
8,leaky05_anchor10_recent,0.5,test,1019,120,0,0.181482,0.265010,0.414633,49.981381,921.676426,0.256797,0.390814,0.249807,0.397765
6,leaky05_anchor10_recent,0.5,train,5420,120,0,0.183614,0.256825,0.427256,51.227696,834.652345,0.330858,0.484156,0.000000,0.000000
7,leaky05_anchor10_recent,0.5,val,1219,120,0,0.145490,0.249319,0.375171,50.131242,969.968251,0.139160,0.352558,0.168442,0.429474
11,leaky07_anchor10_recent,0.7,test,1019,120,0,0.178264,0.266455,0.458828,50.776570,891.708754,0.240680,0.371265,0.228851,0.395850
9,leaky07_anchor10_recent,0.7,train,5420,120,0,0.171498,0.259888,0.458603,52.027094,792.046010,0.306922,0.477741,0.000000,0.000000
10,leaky07_anchor10_recent,0.7,val,1219,120,0,0.169342,0.245698,0.407854,50.936593,903.700270,0.126860,0.316571,0.132225,0.373680
2,raw_anchor10_recent,raw,test,1019,120,0,0.165603,0.259877,0.447752,52.350530,690.775659,0.211849,0.348781,0.177190,0.384596


## 7. Save outputs

In [8]:
out_dir = Path("results/tables")
out_dir.mkdir(parents=True, exist_ok=True)

leaky_results.to_csv(out_dir / "phase2_qrc_leaky_input_probe.csv", index=False)
leaky_diagnostics.to_csv(out_dir / "phase2_qrc_leaky_input_diagnostics.csv", index=False)
leaky_high_vol.to_csv(out_dir / "phase2_qrc_leaky_input_high_vol.csv", index=False)

print("Saved leaky-input QRC tables to", out_dir)

Saved leaky-input QRC tables to results/tables


## Extension list

If leaky input helps but does not close the ESN gap, possible extensions are:

1. delta/momentum encoding with dimension control, e.g. 3 PCA level + 3 PCA delta;
2. streaming/carryover QRC, resetting only at train/val/test split boundaries;
3. extreme-aware readout or residual target against persistence.

Streaming/carryover is intentionally not tested here because the reset-window ESN already strongly outperforms reset-window QRC.